In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

df = pd.read_csv('Task3and4_Loan_Data.csv')
df.tail()

In [ ]:
x = df[['credit_lines_outstanding', 'loan_amt_outstanding',
        'total_debt_outstanding', 'income',
        'years_employed', 'fico_score']]

y = df['default']

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(random_state=42)
rf.fit(x_train, y_train)

In [ ]:
df.iloc[[1]]

In [ ]:
# Probability of default
PD = rf.predict_proba(x_test)[:, 1]

# Convert probabilities to classes only for accuracy/confusion matrix
predicted_default = (PD >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, predicted_default))
print("Confusion matrix:")
print(confusion_matrix(y_test, predicted_default))
print("ROC-AUC:", roc_auc_score(y_test, PD))


def recovery_loss(loan):
    loan_amount = loan['loan_amt_outstanding'].iloc[0]
    rec_rate = 0.1

    Pd = rf.predict_proba(
        loan[['credit_lines_outstanding', 'loan_amt_outstanding',
              'total_debt_outstanding', 'income',
              'years_employed', 'fico_score']]
    )[:, 1][0]

    expected_loss = Pd * (1 - rec_rate) * loan_amount

    return round(float(expected_loss), 5)


recovery_loss(df.iloc[[1]])